# HG4052 · Week 5 Practical
## The same word, twice

**No installs: librosa and numpy are preinstalled on Colab. Headphones on for Part 2: you will hear the recordings you are about to align.**

By the end you will have:
- ✅ an edit-distance grid drawn on paper, then checked against the machine's
- ✅ two recordings of one digit as stacks of MFCC frames, with different frame counts
- ✅ a cost matrix, drawn, with its valley found (your TODO: Week 1's distance line)
- ✅ dynamic time warping, with the one line that is the whole idea written by you
- ✅ a digit recognizer that scores 20 clips it has never heard
- ✅ the same recognizer failing on a new speaker, and your phonetic diagnosis of why
- ✅ (take-home) a WER scorer with S / D / I labels, a k-nearest vote, or a Sakoe–Chiba band

**How this notebook works.** Same as every week: click a cell, press **Shift + Enter**, and read (and hear) the output underneath. Cells marked **✏️ TODO** have one small blank to fill (always one line or less). Every function is provided for you to read and run, never to write. Written TODOs are answered by double-clicking the cell and typing.

**Short on time?** Prioritise **Setup → Part 3 → Part 4 → Part 5**: the cost matrix, the recurrence and the recognizer are the week. Part 1 is a five-minute check of the paper grid, Part 2 is listen-and-count, and Part 7 is take-home by design.

---
### Before this notebook: the paper block

Ten minutes with a pencil, cheat sheet on the slides. Align the two transcriptions of *probably*, careful and casual:

- **Reference** [p ɹ ɑ b ə b l i] (8 symbols) along the top; **casual** [p ɹ ɑ b l i] (6 symbols) down the side.
- Cost of a match 0; of a substitution, insertion or deletion 1.
- Fill the grid with the three-neighbour rule (each cell = its own cost + the smallest of the cell above, the cell to the left and the diagonal), starting from the empty-string corner.
- Trace back from the far corner. Which two segments of the careful form were **deleted** in the casual one?

Keep the grid; Part 1 prints the machine's and you compare.

---
## 0 · Setup

Week 1's commands first: where are you standing, and what is here?

In [ ]:
!pwd
!ls

**Imports.** Nothing new this week: `numpy`, `matplotlib`, `librosa` for loading audio and Week 4's MFCC pipeline, and the `Audio` play button. The only new idea today is one line of arithmetic, and you will write it yourself.

In [ ]:
import os, glob, zipfile, urllib.request, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import librosa
from IPython.display import Audio, display

print("librosa", librosa.__version__, "loaded: the toolbox is open")

**Get today's data.** Two sources, and the cell reports which it found.

1. **The Free Spoken Digit Dataset** (Jakobovski et al., CC BY-SA 4.0): 3,000 recordings of the digits *zero* to *nine* by six speakers (jackson, nicolas, theo, yweweler, george, lucas), fifty of each digit per speaker, 8 kHz, trimmed. Cloned straight from GitHub; about thirty seconds.
2. **The fallback set** from the course repo: 200 clips of the same digits in the same format and naming, spoken by four **synthetic** voices (macOS speech synthesis, five speaking rates each). Used automatically if the clone fails, or set `USE_FSDD = False` to work offline on it.

Both name their files `{digit}_{speaker}_{index}.wav`, so everything below runs on either.

In [ ]:
USE_FSDD = True                                     # False: skip the clone and use the fallback set only
os.makedirs("data", exist_ok=True)

FSDD_DIR = "data/free-spoken-digit-dataset/recordings"
if USE_FSDD and not os.path.isdir(FSDD_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", "-q",
                        "https://github.com/Jakobovski/free-spoken-digit-dataset.git",
                        "data/free-spoken-digit-dataset"], capture_output=True, text=True)
n_fsdd = len(glob.glob(f"{FSDD_DIR}/*.wav"))

FB_ZIP, FB_DIR = "data/fallback_digits.zip", "data/fallback_digits"
if not os.path.exists(FB_ZIP):
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week05/fallback_digits.zip", FB_ZIP)
    except Exception as e:
        print("fallback download failed:", e)
if os.path.exists(FB_ZIP) and not os.path.isdir(FB_DIR):
    with zipfile.ZipFile(FB_ZIP) as z:
        z.extractall("data")
n_fb = len(glob.glob(f"{FB_DIR}/*.wav"))

print(("✅" if n_fsdd >= 3000 else "❌"), f"Free Spoken Digit Dataset: {n_fsdd:,} clips" + ("" if n_fsdd else " (clone skipped or failed)"))
print(("✅" if n_fb >= 200 else "❌"), f"fallback set: {n_fb} clips")

if n_fsdd >= 3000:
    DATA, CLIP_DIR = "fsdd", FSDD_DIR
    SPEAKERS = ["jackson", "nicolas", "theo", "yweweler", "george", "lucas"]
    SPEAKER_A, SPEAKER_B = "jackson", "nicolas"          # change B and re-run Part 6 to try the others
elif n_fb >= 200:
    DATA, CLIP_DIR = "fallback", FB_DIR
    SPEAKERS = ["daniel", "karen", "moira", "aman"]       # four synthetic voices: British, Australian, Irish, Indian English
    SPEAKER_A, SPEAKER_B = "daniel", "karen"
else:
    raise SystemExit("❌ Neither data source is available. The fallback zip is also on NTULearn in the Week 5 folder: "
                     "download it, drag it into data/ via the folder icon in Colab's left sidebar, and re-run this cell.")

def clip_path(digit, speaker, index):
    return f"{CLIP_DIR}/{digit}_{speaker}_{index}.wav"

TEMPLATE_IDX, TEST_IDX = 0, [1, 2]                     # one template per digit; twenty held-out clips (two per digit)
print(f"\nWorking on the {DATA} set. Speaker A = {SPEAKER_A} (templates + held-out tests), speaker B = {SPEAKER_B} (Part 6).")

---
## 1 · The paper grid, checked

The machine fills the same grid you just drew: `D[i, j]` = cost of the best alignment of the first *i* symbols of one string with the first *j* of the other, built from the three neighbours. Read the function once: it is the whole of Part 4 in miniature, on symbols instead of frames.

In [ ]:
def edit_distance(a, b):
    """Minimum edit distance between two symbol lists; returns the table D and the alignment as (a_sym, b_sym) pairs."""
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0] = np.arange(n + 1)                          # deleting everything costs one per symbol
    D[0, :] = np.arange(m + 1)                          # so does inserting everything
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = 0 if a[i - 1] == b[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1,              # from above: delete a[i]
                          D[i, j - 1] + 1,              # from the left: insert b[j]
                          D[i - 1, j - 1] + match)      # diagonal: match or substitute
    # backtrace from the far corner
    i, j, pairs = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (0 if a[i - 1] == b[j - 1] else 1):
            pairs.append((a[i - 1], b[j - 1])); i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + 1:
            pairs.append((a[i - 1], "–")); i -= 1
        else:
            pairs.append(("–", b[j - 1])); j += -1
    return D, pairs[::-1]

careful = list("pɹɑbəbli")
casual  = list("pɹɑbli")
D, pairs = edit_distance(careful, casual)

print("      " + "  ".join(f"{s:>2}" for s in [""] + casual))
for i, row in enumerate(D):
    print(f"{(careful[i-1] if i else ''):>4}  " + "  ".join(f"{v:>2}" for v in row))
print(f"\nedit distance = {D[-1, -1]}")
print("careful :", " ".join(p[0] for p in pairs))
print("casual  :", " ".join(p[1] for p in pairs))
print("deleted :", [p[0] for p in pairs if p[1] == "–"])

**✏️ TODO (in words, double-click).** Does the machine's grid match yours, corner value and path? If a cell differs, find the first one that does: the three-neighbour rule was applied differently there. Which two segments were deleted, and what does the casual form tell you about where reduction strikes in this word?

*Your answer:*

---
## 2 · Digits to MFCCs

Week 4's pipeline, provided as one helper: load at 8 kHz (the dataset's native rate, so nothing lives above 4 kHz), 25 ms frames every 10 ms, 26 mel bands, 13 MFCCs, plus their deltas and delta-deltas: **39 numbers per frame**, exactly what the lecture said a recognizer hears. Each clip becomes a matrix with one row per frame.

In [ ]:
SR = 8000                                              # the dataset's rate; frames are 25 ms (200 samples), hop 10 ms (80)

def mfcc_frames(path, deltas=True):
    """One clip → (frames, 39) matrix: 13 MFCCs + 13 deltas + 13 delta-deltas per 10 ms frame."""
    y, _ = librosa.load(path, sr=SR)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=13, n_fft=256, win_length=200, hop_length=80,
                             n_mels=26, fmax=4000)                     # (13, frames)
    if deltas:
        w = 9 if m.shape[1] >= 9 else 3                                # delta window, shrunk for very short clips
        m = np.vstack([m, librosa.feature.delta(m, width=w), librosa.feature.delta(m, order=2, width=w)])
    return m.T                                                         # (frames, 39)

seven_a = clip_path(7, SPEAKER_A, 0)
seven_b = clip_path(7, SPEAKER_A, 1)
for p in (seven_a, seven_b):
    y, _ = librosa.load(p, sr=SR)
    print(f"{os.path.basename(p):<22} {len(y)/SR:.2f} s  →  {mfcc_frames(p).shape[0]} frames of 39 numbers")
    display(Audio(y, rate=SR))

**✏️ TODO (in words, double-click).** Same speaker, same word, and the two frame counts differ. Give two reasons, one about the speaker and one about the recording.

*Your answer:*

---
## 3 · The cost matrix

Lay recording A's frames along one axis and B's along the other. Cell (i, j) holds the distance between frame *i* of A and frame *j* of B: a small number where the two frames sound alike, a large one where they do not. The only arithmetic in it is Week 1's line, and that line is yours.

In [ ]:
def frame_distance(a, b):
    # ✏️ TODO: the Euclidean distance between two 39-number frames: square the differences,
    # add them up, take the square root. Week 1's line, or numpy's one-word version of it.
    # Shape of the answer:   np.sqrt(np.sum((a - b) ** 2))      or      np.linalg.norm(a - b)
    return ...

def cost_matrix(A, B):
    """(nA, nB) matrix of frame-to-frame distances."""
    return np.array([[frame_distance(a, b) for b in B] for a in A])

if frame_distance(np.zeros(3), np.ones(3)) is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    check = frame_distance(np.array([0., 0., 0.]), np.array([1., 2., 2.]))
    assert abs(check - 3.0) < 1e-9, "Expected 3.0 for (0,0,0) vs (1,2,2): sqrt(1 + 4 + 4)"
    print("✅ frame_distance works: (0,0,0) vs (1,2,2) →", check)

In [ ]:
if frame_distance(np.zeros(3), np.ones(3)) is ...:
    print("⬆ fill the TODO in the cell above first")
else:
    A = mfcc_frames(seven_a)
    B_same = mfcc_frames(seven_b)                      # another 'seven'
    B_diff = mfcc_frames(clip_path(1, SPEAKER_A, 1))   # a 'one'
    C_same, C_diff = cost_matrix(A, B_same), cost_matrix(A, B_diff)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    vmax = max(C_same.max(), C_diff.max())
    for ax, C, title in zip(axes, (C_same, C_diff), ("seven vs seven: look for the valley", "seven vs one: no valley")):
        im = ax.imshow(C.T, origin="lower", aspect="auto", cmap="magma", vmax=vmax)
        ax.set_xlabel("frames of A (seven)"); ax.set_ylabel("frames of B"); ax.set_title(title)
    fig.colorbar(im, ax=axes, label="frame distance", shrink=0.8)
    plt.show()
    print(f"mean cell cost, same word: {C_same.mean():.1f}   different word: {C_diff.mean():.1f}")

**✏️ TODO (in words, double-click).** In the same-word matrix, the dark valley runs roughly corner to corner. Where it runs **horizontally** for a stretch (one frame of A staying cheap against several frames of B), what happened phonetically in recording B? And what would a valley that is dark but far from the diagonal mean?

*Your answer:*

---
## 4 · The DTW recurrence: one line

The best alignment ending at cell (i, j) must have entered it from the cell **above**, the cell to the **left**, or the **diagonal**; there is no other way in. So the best total cost to reach (i, j) is the cell's own cost plus the cheapest of those three totals. Fill the table row by row and every one of the astronomically many paths has been priced. That sentence is the lecture; the line below is the sentence in numpy.

The table has one extra row and column (index 0) standing for "nothing aligned yet": cost 0 at the origin, impossible (infinite) elsewhere, so the first real row and column can only come from the origin or along the edge.

In [ ]:
def dtw(cost):
    """cost: (n, m) matrix. Returns (total cost, cumulative table D, optimal path as a list of (i, j))."""
    n, m = cost.shape
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # ✏️ TODO: this cell's cost plus the cheapest of the three ways in.
            # Building blocks:   cost[i-1, j-1]   min(...)   D[i-1, j]  (above)   D[i, j-1]  (left)   D[i-1, j-1]  (diagonal)
            D[i, j] = ...
    return D[n, m], D, backtrace(D)

def backtrace(D):
    """Walk from the far corner back to the origin, always to the cheapest of the three neighbours."""
    i, j = D.shape[0] - 1, D.shape[1] - 1
    path = [(i - 1, j - 1)]
    while (i, j) != (1, 1):
        moves = {(i - 1, j - 1): D[i - 1, j - 1], (i - 1, j): D[i - 1, j], (i, j - 1): D[i, j - 1]}
        i, j = min(moves, key=moves.get)
        path.append((i - 1, j - 1))
    return path[::-1]

def dtw_reference(cost):
    """The finished recurrence, for the fallback below (and to read if you are stuck: it IS the answer)."""
    n, m = cost.shape
    D = np.full((n + 1, m + 1), np.inf); D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = cost[i - 1, j - 1] + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return D[n, m], D, backtrace(D)

# the lecture's tiny example: A = [1, 3, 4, 2], B = [1, 2, 4, 4, 2], cost = |a - b|, best total 1
toy = np.abs(np.subtract.outer([1, 3, 4, 2], [1, 2, 4, 4, 2])).astype(float)
try:
    total, D_toy, path_toy = dtw(toy)
    TODO_DONE = np.isfinite(total)
except TypeError:
    TODO_DONE = False

if not TODO_DONE:
    print("⬆ fill the TODO first, then re-run. (Parts 5 to 7 will run on dtw_reference until you do.)")
    DTW = dtw_reference
else:
    assert abs(total - 1.0) < 1e-9, f"The lecture's 4×5 example should cost exactly 1, got {total}"
    ex = np.abs(np.subtract.outer([1, 2], [1, 3])).astype(float)       # exercise 5.2: A = [1, 2], B = [1, 3]
    assert abs(dtw(ex)[0] - 1.0) < 1e-9, "Exercise 5.2 should cost 1 (a single diagonal step)"
    DTW = dtw
    print("✅ Your recurrence reproduces the lecture's table: total cost 1, path", path_toy)
    print("✅ and exercise 5.2: total cost 1. One line, and the exponential zoo of alignments is priced.")

In [ ]:
if frame_distance(np.zeros(3), np.ones(3)) is ...:
    print("⬆ Part 3's TODO first")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, C, title in zip(axes, (C_same, C_diff), ("seven vs seven", "seven vs one")):
        total, D_, path = DTW(C)
        ax.imshow(C.T, origin="lower", aspect="auto", cmap="magma", vmax=vmax)
        ax.plot([p[0] for p in path], [p[1] for p in path], color="#0e6f66", lw=2.5)
        ax.set_title(f"{title}: DTW cost {total:.0f} over {len(path)} steps = {total/len(path):.1f} per step")
        ax.set_xlabel("frames of A"); ax.set_ylabel("frames of B")
    plt.show()
    print("The path hugs the valley when the words match and wanders when they do not;")
    print("the per-step cost is what the recognizer in Part 5 compares.")

**✏️ TODO (by hand, then check).** A = [1, 2, 3] and B = [2, 2, 4], cost = |a − b|. Fill the 3 × 3 table on paper with the three-neighbour rule, write down the total and the path, then run the cell below to check. Where does the path leave the diagonal, and why?

*Your total and path:*

In [ ]:
toy3 = np.abs(np.subtract.outer([1, 2, 3], [2, 2, 4])).astype(float)
total3, D3, path3 = DTW(toy3)
print("cost matrix:\n", toy3.astype(int))
print("cumulative table D (with the index-0 border):\n", np.where(np.isinf(D3), -1, D3).astype(int), "   (-1 = impossible)")
print(f"total = {total3:.0f}, path = {path3}")

---
## 5 · A recognizer, from ten templates

Store one clip per digit from speaker A (index 0) as the template. To recognize a new clip, align it with all ten templates and pick the cheapest. The score is the DTW cost per step, so long clips are not punished for being long. Twenty held-out clips of speaker A (indices 1 and 2 of each digit: **never the clips you stored**) are the test. The loop and the confusion matrix are provided; the DTW inside them is yours.

In [ ]:
templates = {d: mfcc_frames(clip_path(d, SPEAKER_A, TEMPLATE_IDX)) for d in range(10)}

def recognize(frames):
    scores = {}
    for d, T in templates.items():
        total, _, path = DTW(cost_matrix(frames, T))
        scores[d] = total / len(path)
    return min(scores, key=scores.get), scores

def evaluate(speaker, indices=TEST_IDX, verbose=True):
    confusion = np.zeros((10, 10), dtype=int)
    t0 = time.time()
    for d in range(10):
        for k in indices:
            pred, _ = recognize(mfcc_frames(clip_path(d, speaker, k)))
            confusion[d, pred] += 1
    acc = np.trace(confusion) / confusion.sum()
    if verbose:
        print(f"speaker {speaker}: {np.trace(confusion)} of {confusion.sum()} correct = {acc:.0%}   ({time.time()-t0:.1f} s)")
    return acc, confusion

def show_confusion(confusion, title):
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(confusion, cmap="Blues")
    for i in range(10):
        for j in range(10):
            if confusion[i, j]:
                ax.text(j, i, confusion[i, j], ha="center", va="center", color="white" if confusion[i, j] > 1 else "black")
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xlabel("recognized as"); ax.set_ylabel("spoken digit"); ax.set_title(title)
    plt.show()

if frame_distance(np.zeros(3), np.ones(3)) is ...:
    print("⬆ Part 3's TODO first")
else:
    acc_A, conf_A = evaluate(SPEAKER_A)
    show_confusion(conf_A, f"speaker A ({SPEAKER_A}), held-out clips")
    wrong = [(d, p) for d in range(10) for p in range(10) if d != p and conf_A[d, p]]
    print("confusions (spoken → recognized):", wrong if wrong else "none")

**✏️ TODO (in words, double-click).** Pick one confusion from the matrix (if speaker A made none, run `evaluate(SPEAKER_A, indices=[3, 4, 5])` for a harder set, or take one from Part 6). Explain it phonetically: which sounds do the two digits share, and which distinguishing sound is short, weak or easily lost in a 39-number frame?

*Your answer:*

---
## 6 · Break it: a new speaker

Same ten templates, speaker B's voice. Watch the accuracy move, then diagnose it in terms you own.

In [ ]:
if frame_distance(np.zeros(3), np.ones(3)) is ...:
    print("⬆ Part 3's TODO first")
else:
    acc_B, conf_B = evaluate(SPEAKER_B)
    show_confusion(conf_B, f"speaker B ({SPEAKER_B}) against speaker A's templates")
    print(f"accuracy: speaker A {acc_A:.0%}  →  speaker B {acc_B:.0%}")
    wrong_B = [(d, p) for d in range(10) for p in range(10) if d != p and conf_B[d, p]]
    print("confusions (spoken → recognized):", wrong_B if wrong_B else "none")
    print("Other speakers to try:", [s for s in SPEAKERS if s not in (SPEAKER_A, SPEAKER_B)], "(set SPEAKER_B and re-run)")

**✏️ TODO (three sentences, double-click).** Diagnose the drop phonetically: vowel spaces (a different speaker's /i/ or /ɑ/ sits elsewhere in F1–F2, so every frame distance grows), rate (DTW fixes timing, so rate alone should not hurt: does the evidence agree?), voice quality or accent. Then: would storing more templates from speaker A fix it, or does the fix need speaker B's voice in the templates?

*Your answer:*

---
## 7 · Stretch (take-home, pick one)

Three directions, each a few lines. All three run on the fallback set as well as the real one.

**(a) Word error rate with S / D / I labels.** Edit distance again, on words. The lecture's example first, then any pair you like. WER = (S + D + I) / reference length, and it can exceed 100%.

In [ ]:
def wer(reference, hypothesis):
    ref, hyp = reference.split(), hypothesis.split()
    D, pairs = edit_distance(ref, hyp)
    labels = ["S" if r != h and r != "–" and h != "–" else "D" if h == "–" else "I" if r == "–" else "="
              for r, h in pairs]
    S, Dl, I = labels.count("S"), labels.count("D"), labels.count("I")
    return (S + Dl + I) / len(ref), labels, pairs

for ref, hyp in [("the dog ran home", "the dog run home fast"),          # exercise 5.1: expect 50%
                 ("I am gotta go", "I am going to go"),
                 ("recognize speech", "wreck a nice beach")]:
    rate, labels, pairs = wer(ref, hyp)
    print(f"{ref!r} vs {hyp!r}: WER = {rate:.0%}   " + " ".join(f"{r}/{h}:{l}" for (r, h), l in zip(pairs, labels)))

**(b) k-nearest templates.** Store several clips per digit from speaker A and let them vote. Does more of A's voice rescue speaker B, or only A's own held-out clips? (Run time grows with k.)

In [ ]:
def evaluate_knn(speaker, k=3, indices=TEST_IDX):
    bank = [(d, mfcc_frames(clip_path(d, SPEAKER_A, i))) for d in range(10) for i in range(k)]
    correct = 0
    for d in range(10):
        for t in indices:
            if t < k: continue                                           # never test on a stored clip
            F = mfcc_frames(clip_path(d, speaker, t))
            scored = sorted((DTW(cost_matrix(F, T))[0] / len(F), lab) for lab, T in bank)
            votes = [lab for _, lab in scored[:k]]
            correct += (max(set(votes), key=votes.count) == d)
    n = sum(1 for t in indices if t >= k) * 10
    return correct / n if n else float("nan")

if frame_distance(np.zeros(3), np.ones(3)) is not ...:
    for spk in (SPEAKER_A, SPEAKER_B):
        print(f"{spk}: 1 template {evaluate(spk, indices=[3, 4], verbose=False)[0]:.0%}   "
              f"3 templates, vote {evaluate_knn(spk, k=3, indices=[3, 4]):.0%}")

**(c) The Sakoe–Chiba band.** Real alignments never stray far from the diagonal, so forbid cells further than `band` frames from it: fewer cells to fill, and a guard against silly paths. Time it, and check whether accuracy survives.

In [ ]:
def dtw_banded(cost, band):
    n, m = cost.shape
    D = np.full((n + 1, m + 1), np.inf); D[0, 0] = 0.0
    for i in range(1, n + 1):
        jc = i * m / n                                                   # where the diagonal is on this row
        for j in range(max(1, int(jc - band)), min(m, int(jc + band)) + 1):
            D[i, j] = cost[i - 1, j - 1] + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return D[n, m], D, backtrace(D)

if frame_distance(np.zeros(3), np.ones(3)) is not ...:
    for band in (None, 10, 5):
        DTW = dtw_reference if band is None else (lambda c, b=band: dtw_banded(c, b))
        t0 = time.time(); acc, _ = evaluate(SPEAKER_A, verbose=False)
        print(f"band {str(band):>4}: accuracy {acc:.0%}, {time.time()-t0:.1f} s for 20 clips")
    DTW = dtw if TODO_DONE else dtw_reference

---
**Done.** You built the recognizer that opened the field: stored examples, a distance, and one line that lets time bend. Next week the distance becomes a probability and the template becomes a model of how a word *can* sound.